In [1]:
import pickle
import pandas as pd
import numpy as np

In [2]:
blind = {33, 35, 36, 38, 39, 41, 42, 43, 53}
ctrlA = {3, 4, 5, 6, 7, 8, 9, 10, 11, 27}
ctrlAV = {12, 13, 14, 15, 16, 17, 18, 19, 22, 32}

participant_to_group = {}

for p in blind:
    participant_to_group[p] = "blind"
for p in ctrlA:
    participant_to_group[p] = "ctrlA"
for p in ctrlAV:
    participant_to_group[p] = "ctrlAV"

run_start_indices = [0, 267, 492, 812, 1137, 1373]

In [3]:
def convert_nested_dict_to_df(data_dict, participant_to_group):
    df_list = []
    
    for participant, roi_dict in data_dict.items():
        for roi, model_dict in roi_dict.items():

            temp = (
                pd.DataFrame(model_dict)
                .reset_index()
                .melt(id_vars="index", var_name="model", value_name="value")
            )

            temp["run"] = np.searchsorted(run_start_indices, temp["index"], side="right")

            temp["participant"] = participant
            temp["roi"] = roi
            temp["roi_type"] = temp["roi"].apply(lambda x: "language" if isinstance(x, (int, float)) else "visual")
            temp["participant_group"] = participant_to_group[participant]

            df_list.append(temp)
    return pd.concat(df_list, ignore_index=True)

In [6]:
with open("/Users/pien/repos/paper/results/january/groundft/results.pkl", "rb") as f:
    data = pickle.load(f)

df = convert_nested_dict_to_df(data, participant_to_group)


In [7]:
df

,index,model,value,run,participant,roi,roi_type,participant_group
0,0,fast_text_grounded,0.078714,1,33,2,language,blind
1,1,fast_text_grounded,0.092351,1,33,2,language,blind
2,2,fast_text_grounded,0.171963,1,33,2,language,blind
3,3,fast_text_grounded,0.213461,1,33,2,language,blind
4,4,fast_text_grounded,0.121987,1,33,2,language,blind
...,...,...,...,...,...,...,...,...
228660,1572,fast_text_grounded,-0.080672,6,32,visual,visual,ctrlAV
228661,1573,fast_text_grounded,-0.003236,6,32,visual,visual,ctrlAV
228662,1574,fast_text_grounded,0.041847,6,32,visual,visual,ctrlAV
228663,1575,fast_text_grounded,0.048680,6,32,visual,visual,ctrlAV


In [8]:
with open("/Users/pien/repos/paper/results/january/groundft/results_semantic.pkl", "rb") as f:
    semanticresults = pickle.load(f)

print(semanticresults)

{('binder_concreteness', 'fast_text_grounded'): array([-0.00598565, -0.06844706,  0.00322578, ...,  0.07606449,
        0.06997101,  0.36794157], shape=(1577,)), ('binder_abstractness', 'fast_text_grounded'): array([ 0.155712  , -0.12857018, -0.05964612, ...,  0.04848305,
        0.0443158 ,  0.34581643], shape=(1577,))}


In [9]:
semantic_models = set(m for _, m in semanticresults.keys())
df_models = set(df["model"].unique())

print("Semantic-only models:", semantic_models - df_models)
print("DF-only models:", df_models - semantic_models)


Semantic-only models: set()
DF-only models: set()


In [10]:
def add_semantic_results(df, semantic):
    df = df.copy()
    df["corr_concreteness"] = np.nan
    df["corr_abstractness"] = np.nan

    for (binder, semantic_model), arr in semantic.items():
        mask = df["model"] == semantic_model
        if not mask.any():
            continue

        col = (
            "corr_concreteness"
            if binder == "binder_concreteness"
            else "corr_abstractness"
        )

        idx = df.loc[mask, "index"].values
        valid = idx < len(arr)

        df.loc[mask[mask].index[valid], col] = arr[idx[valid]]

    return df


In [11]:
final_df = add_semantic_results(df, semanticresults)

In [12]:
final_df

,index,model,value,run,participant,roi,roi_type,participant_group,corr_concreteness,corr_abstractness
0,0,fast_text_grounded,0.078714,1,33,2,language,blind,-0.005986,0.155712
1,1,fast_text_grounded,0.092351,1,33,2,language,blind,-0.068447,-0.128570
2,2,fast_text_grounded,0.171963,1,33,2,language,blind,0.003226,-0.059646
3,3,fast_text_grounded,0.213461,1,33,2,language,blind,0.057307,-0.022814
4,4,fast_text_grounded,0.121987,1,33,2,language,blind,0.083167,-0.020599
...,...,...,...,...,...,...,...,...,...,...
228660,1572,fast_text_grounded,-0.080672,6,32,visual,visual,ctrlAV,0.078790,0.050805
228661,1573,fast_text_grounded,-0.003236,6,32,visual,visual,ctrlAV,0.076980,0.049340
228662,1574,fast_text_grounded,0.041847,6,32,visual,visual,ctrlAV,0.076064,0.048483
228663,1575,fast_text_grounded,0.048680,6,32,visual,visual,ctrlAV,0.069971,0.044316


In [13]:
nan_rows = df[df[["corr_concreteness", "corr_abstractness"]].isna().any(axis=1)]




KeyError: "None of [Index(['corr_concreteness', 'corr_abstractness'], dtype='object')] are in the [columns]"

In [ ]:
df[df[["value"]].isna().any(axis=1)]["model"].unique()

In [ ]:
nan_rows

In [14]:
final_df.to_csv("january/groundft/results.csv", sep=";", index=False)